# US flight cancellations — quick EDA

Class imbalance, per-carrier cancellation rates, and a peek at the feature distributions on a single year of data. The full pipeline (10-year corpus, PySpark prep, custom + MLlib LR) runs from `python -m src.pipeline`.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import matplotlib.pyplot as plt
import pandas as pd

from src import config

## Single-year snapshot

Loading the full 10-year corpus into pandas would blow up memory — that's why the production pipeline uses Spark. For EDA we look at one year.

In [ ]:
df = pd.read_csv(config.DATA_DIR / '2018.csv', usecols=list(config.COLUMNS_TO_KEEP))
print(f'{len(df):,} flights in 2018')
df.head()

## Class distribution — heavily imbalanced

Roughly 1.5–2% cancellation rate. The production pipeline undersamples the majority class before training so the binary classifier doesn't degenerate.

In [ ]:
rate = df['CANCELLED'].mean()
print(f'Cancellation rate in 2018: {rate:.2%}')
df['CANCELLED'].value_counts().plot.bar(figsize=(6, 3), title='Flights by CANCELLED label')
plt.tight_layout()
plt.show()

## Cancellation rate by carrier

Some operators are noticeably more cancellation-prone than others — the trained classifier picks this up via the `AIRLINE_ID` feature (`StringIndexer`-encoded `OP_CARRIER`).

In [ ]:
by_carrier = (
    df.groupby('OP_CARRIER')['CANCELLED']
    .agg(['mean', 'count'])
    .rename(columns={'mean': 'cancel_rate', 'count': 'flights'})
    .sort_values('cancel_rate', ascending=False)
)
by_carrier